## ESPnet TTS Demonstracija

### 0. Instaliuojame bibliotekas

In [ ]:
# !apt-get update
!pip install "espnet[tts] @ git+https://github.com/airenas/espnet.git@demo.v01" phonemizer -q --disable-pip-version-chec
!pip install parallel_wavegan -q --disable-pip-version-check

print("\nDONE: paruošta")

### 0.1 Pataisome PWGan dėl Python 3.12

In [1]:
## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser

### 1. Inicializuojame ESPnet ir pagalbines funkcijas

In [2]:
from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from huggingface_hub import snapshot_download
from pathlib import Path

device = "cpu"
# device = "cuda" # uncomment if you have a GPU and want to use it

def download_vocoder(tag: str) -> str:
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)

    for file in repo_path.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError("No *.pkl file found in the vocoder repo")

class TTSData:
    def __init__(self, tts, info):
        self.tts = tts
        self.info = info


def init_tts(am_tag=None, vocoder_tag=None, info=""):
    voc_str = vocoder_tag
    if not voc_str:
        voc_str = "Griffin-Lim"

    print(f"\n===================================\nAM      = {am_tag}")
    print(f"Vocoder = {voc_str}")
    print(f"===================================")

    vocoder_file=None
    if vocoder_tag:
        vocoder_file = download_vocoder(vocoder_tag)
        vocoder_tag = None

    tts = Text2Speech.from_pretrained(
        model_tag=am_tag,
        vocoder_tag=vocoder_tag,
        vocoder_file=vocoder_file,
        device=device
    )
    return TTSData(tts, info)

print ("\nDONE: ESPnet paruoštas")

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'

DONE: ESPnet paruoštas


In [3]:
from huggingface_hub import login

login()

### 2. Parsiunčiame/užkrauname modelius iš HuggingFace

Paruošti tokie modeliai:
1. AM vyriškas balsas: [VSSA-SDSA/sing-arn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-arn.fastspeech2.v01)
2. AM moteriškas balsas - [VSSA-SDSA/sing-agn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01)
4. Vokoderis vyriškas balsas - [VSSA-SDSA/sing-arn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-arn.vocoder.style_melgan.v01)
5. Vokoderis moteriškas balsas - [VSSA-SDSA/sing-agn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01)


In [4]:
am_tag_hf="airenas/sing-sau.fastspeech2.v01"
vocoder_tag_hf= "airenas/sing-sau.vocoder.style_melgan.v01"

tts_hf_v1 = init_tts(am_tag=am_tag_hf, vocoder_tag=vocoder_tag_hf, info="ner-fastspeech2+ner-style")

tts_list = [tts_hf_v1]

print (f"\nDONE: {len(tts_list)} modeliai paruošti\n")


AM      = airenas/sing-sau.fastspeech2.v01
Vocoder = airenas/sing-sau.vocoder.style_melgan.v01


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 28 files:   0%|          | 0/28 [00:00<?, ?it/s]


DONE: 1 modeliai paruošti



/home/airenas/miniconda3/envs/espnet/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


### 3. Rezultatas

Sintezuojamas tekstas

In [5]:
texts = ["sil \"i S p r a dZ' ^iu: b \"u v o: v' \"ie n. sp \"a: m.' Z' i n a s , b' e r' \"i b' i s , t a m. s \"u s x a \"o s a s sp g' i: v' \"i: b' E: S a l.' t' \"i n' i s . sil",
        "sil v' \"i s o: S' \"i t o: s v ai z d ^u: , m' i n.' tS' ^iu: sp i ^r.' j eu s m ^u: n \"uo t r u p o: s sp t ^ai s u s m u l.' k' \"E: d a v o: , t ^ai v' ^E: l. i S \"au g d a v o: ^i: j \"eu d' i n a n.' tS' e z d r a m ^a: t' i S k a s' ts' e n \"a s . sil b' \"e ^a: b' e j io: , l' \"iu ts' E: s p a d a r' \"i: t a s' \"i: s p u: d' i s sp b \"u v o: s' t' i p' r' \"eu s' e s sp \"i S' v' \"i s o: a t \"o: s t o: g u: sp g' i: v' ^e: n' i m o: . sil",
        "sil b' \"e t ^o: , tS' \"e b ^o: d \"a: r. sp l' \"iu d o: d r ^au g o: k l' ^ie r' i k o: p' e t' r' ^i: l o: s t' \"E: v' i S' k' E: . sil n u v a Z' ^e: v o: j ^ie d u \"i S v ^a: k a r o: sp ^i: p' i r. m \"uo s' iu s m' i S p a r \"u s . sil",
        "sil t ^uo m' e t \"u S' \"i s k l \"au s' i m a z b \"u v o: l a b ^ai a S t r \"u s , i ^r. p a tS' ^io: s' e k l' e b o: n' \"i j io: s' e t a ^r. p k u n' i g ^u: k' \"i l. d a v o: sp i: v ai r' ^iu: i n.' ts' i d' e ^n. t u: . sil",
        "sil k l' ^ie r' i k a s v a s ^a: r' i s t ^ai p' i ^r. p a s' i l' \"i k o: b' e s t o: v' ^i: s , s u p l \"o: t a z g' \"E: d o: s' i ^r. s u n' ^ie k' i n. t a z d' ^e: S' i m. t k \"a: r. t u: d au g' ^eu n' e g \"u a n ^uo m' e t , p' e ^r. ^a: t l ai d u s . sil S' \"i t o: k' io: s a p' l' i N.' k' \"i: b' E: s t ^ai j \"i s' n' e b \"u v o: n u m ^a: t' e: s , r ^uo Z d a m a s' i s' ^i: a ^n. t r a: j i: s u s' i t' i k' \"i m a: s \"u l' \"iu ts' e . sil",
    ]
for i, text in enumerate(texts):
    print(f"\n==========================================================================\nText = {text}")
    for it, tts_data in enumerate(tts_list):
        tts = tts_data.tts
        wav = tts(text)["wav"]
        filename = f"output_{i}_{it}.wav"
        sf.write(filename, wav.cpu().numpy(), tts.fs)
        audio_widget = Audio(filename)._repr_html_()
        display(HTML(f"<div style='display:flex; align-items:center; gap:10px;'>{audio_widget}<span>{tts_data.info}:</span></div>"))



Text = sil "i S p r a dZ' ^iu: b "u v o: v' "ie n. sp "a: m.' Z' i n a s , b' e r' "i b' i s , t a m. s "u s x a "o s a s sp g' i: v' "i: b' E: S a l.' t' "i n' i s . sil



Text = sil v' "i s o: S' "i t o: s v ai z d ^u: , m' i n.' tS' ^iu: sp i ^r.' j eu s m ^u: n "uo t r u p o: s sp t ^ai s u s m u l.' k' "E: d a v o: , t ^ai v' ^E: l. i S "au g d a v o: ^i: j "eu d' i n a n.' tS' e z d r a m ^a: t' i S k a s' ts' e n "a s . sil b' "e ^a: b' e j io: , l' "iu ts' E: s p a d a r' "i: t a s' "i: s p u: d' i s sp b "u v o: s' t' i p' r' "eu s' e s sp "i S' v' "i s o: a t "o: s t o: g u: sp g' i: v' ^e: n' i m o: . sil



Text = sil b' "e t ^o: , tS' "e b ^o: d "a: r. sp l' "iu d o: d r ^au g o: k l' ^ie r' i k o: p' e t' r' ^i: l o: s t' "E: v' i S' k' E: . sil n u v a Z' ^e: v o: j ^ie d u "i S v ^a: k a r o: sp ^i: p' i r. m "uo s' iu s m' i S p a r "u s . sil



Text = sil t ^uo m' e t "u S' "i s k l "au s' i m a z b "u v o: l a b ^ai a S t r "u s , i ^r. p a tS' ^io: s' e k l' e b o: n' "i j io: s' e t a ^r. p k u n' i g ^u: k' "i l. d a v o: sp i: v ai r' ^iu: i n.' ts' i d' e ^n. t u: . sil



Text = sil k l' ^ie r' i k a s v a s ^a: r' i s t ^ai p' i ^r. p a s' i l' "i k o: b' e s t o: v' ^i: s , s u p l "o: t a z g' "E: d o: s' i ^r. s u n' ^ie k' i n. t a z d' ^e: S' i m. t k "a: r. t u: d au g' ^eu n' e g "u a n ^uo m' e t , p' e ^r. ^a: t l ai d u s . sil S' "i t o: k' io: s a p' l' i N.' k' "i: b' E: s t ^ai j "i s' n' e b "u v o: n u m ^a: t' e: s , r ^uo Z d a m a s' i s' ^i: a ^n. t r a: j i: s u s' i t' i k' "i m a: s "u l' "iu ts' e . sil
